# Load and process Klebsiella pneumoniae data

In [ ]:
import numpy as np
import pandas as pd
from Bio import AlignIO, Phylo
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load and check `.aln` data

In [ ]:
seq_path = "../data/klebsiella/noref.aln"
alignment = AlignIO.read(seq_path, "fasta")

print("Alignment summary:")
print(alignment)
print("-" * 40)

seq_count = 0
seq_names = []

for record in alignment:
    print(record.id, len(record.seq))
    seq_names.append(record.id)
    seq_count += 1

print(f"Total number of sequences: {seq_count}")

## Load `.nwk` tree

In [ ]:
clonal_tree = Phylo.read("../data/klebsiella/klebsiella_clonal.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

In [ ]:
clonal_edge = np.loadtxt("../data/klebsiella/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/klebsiella/clonal_node_height.csv", delimiter=",", dtype=float)

In [ ]:
clonal_edge, clonal_node_height

## Load genome data and transfer to boolean matrix

In [ ]:
leaf_names = np.loadtxt("../data/klebsiella/tip_names.csv", delimiter=",", dtype=str)
leaf_names = np.char.strip(leaf_names, '"')
len(leaf_names), len(seq_names)

In [ ]:
# sequences = []

# for record in alignment:
#     seq_chars = list(str(record.seq).upper())
#     sequences.append(seq_chars)

# char_matrix = np.array(sequences)

# reference_seq = char_matrix[0]
# bool_mat = (char_matrix != reference_seq)

# bool_mat.shape, bool_mat.dtype

In [ ]:
# index_map = {string_id: idx for idx, string_id in enumerate(seq_names)}
# new_indices = [index_map[string_id] for string_id in leaf_names]
# genomes_bool = bool_mat[new_indices]
# genomes_bool.shape, genomes_bool.dtype

In [ ]:
# np.savetxt("../data/klebsiella/genomes_bool.csv", genomes_bool, delimiter=",", fmt="%d")
bool_mat = np.loadtxt("../data/klebsiella/genomes_bool.csv", delimiter=",", dtype=bool)
bool_mat.shape, bool_mat.dtype

## Compute estimated theta

In [ ]:
has_true = bool_mat.any(axis=0)
has_false = ~bool_mat.all(axis=0)
idx_seg = np.where(has_true & has_false)[0]

In [ ]:
count_S = idx_seg.size
count_S

In [ ]:
bool_mat.shape[1]

In [ ]:
np.sum(clonal_edge[:, 2])

theta / 2 = S / L / tree length

In [ ]:
print("theta:", float(count_S / bool_mat.shape[1] / np.sum(clonal_edge[:, 2]) * 2))

## Get summary statistics for random genome segments with fixed length

Randomly select 2000 segments with windows of 1000, 4000, 8000 bp each.

In [ ]:
np.random.seed(100)
clonal_tree = ClonalTree(n=100)

clonal_tree.edge = clonal_edge
clonal_tree.node_height = clonal_node_height
clonal_tree.height = np.max(clonal_node_height)
clonal_tree.length = np.sum(clonal_edge[:, 2])

In [ ]:
rand_seg_df = pd.DataFrame({'Gene_Length': np.hstack([np.repeat(1000, 2000), np.repeat(4000, 2000), np.repeat(8000, 2000)])})
rand_seg_df['Start_pos'] = None
rand_seg_df['End_pos'] = None
rand_seg_df['Alignment'] = True
rand_seg_df

In [ ]:
np.random.seed(100)

max_start_pos = bool_mat.shape[1] - 1000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 1000 - 1
    rand_seg_df.loc[i, 'Start_pos'] = start_pos
    rand_seg_df.loc[i, 'End_pos'] = end_pos

max_start_pos = bool_mat.shape[1] - 4000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 4000 - 1
    rand_seg_df.loc[2000+i, 'Start_pos'] = start_pos
    rand_seg_df.loc[2000+i, 'End_pos'] = end_pos

max_start_pos = bool_mat.shape[1] - 8000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 8000 - 1
    rand_seg_df.loc[4000+i, 'Start_pos'] = start_pos
    rand_seg_df.loc[4000+i, 'End_pos'] = end_pos

print(rand_seg_df.head())
rand_seg_df.to_csv("../data/klebsiella/rand_seg_info.csv", index=False)

In [ ]:
rand_seg1000_summary = np.full((2000, 46), np.nan)
rand_seg4000_summary = np.full((2000, 46), np.nan)
rand_seg8000_summary = np.full((2000, 46), np.nan)
rand_seg1000_summary.shape, rand_seg4000_summary.shape, rand_seg8000_summary.shape

In [ ]:
np.random.seed(100)

max_start_pos = bool_mat.shape[1] - 1000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 1000 - 1
    rand_seg_df.loc[i, 'Start_pos'] = start_pos
    rand_seg_df.loc[i, 'End_pos'] = end_pos
    
    seg_matrix = bool_mat[:, start_pos-1:end_pos]  # Adjust for 0-based indexing
    summary_stats = segment_summary_stats(clonal_tree, seg_matrix)
    rand_seg1000_summary[i, :] = summary_stats

In [ ]:
max_start_pos = bool_mat.shape[1] - 4000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 4000 - 1
    rand_seg_df.loc[2000+i, 'Start_pos'] = start_pos
    rand_seg_df.loc[2000+i, 'End_pos'] = end_pos
    
    seg_matrix = bool_mat[:, start_pos-1:end_pos]  # Adjust for 0-based indexing
    summary_stats = segment_summary_stats(clonal_tree, seg_matrix)
    rand_seg4000_summary[i, :] = summary_stats

In [ ]:
max_start_pos = bool_mat.shape[1] - 8000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    end_pos = start_pos + 8000 - 1
    rand_seg_df.loc[4000+i, 'Start_pos'] = start_pos
    rand_seg_df.loc[4000+i, 'End_pos'] = end_pos
    
    seg_matrix = bool_mat[:, start_pos-1:end_pos]  # Adjust for 0-based indexing
    summary_stats = segment_summary_stats(clonal_tree, seg_matrix)
    rand_seg8000_summary[i, :] = summary_stats

In [ ]:
rand_seg_df.to_csv("../data/stklebsiellaaph/rand_seg_info.csv", index=False)

In [ ]:
np.savetxt("../data/klebsiella/rand_seg1000.csv", rand_seg1000_summary, delimiter=",")
np.savetxt("../data/klebsiella/rand_seg2000.csv", rand_seg4000_summary, delimiter=",")
np.savetxt("../data/klebsiella/rand_seg8000.csv", rand_seg8000_summary, delimiter=",")

## Get summary statistics for random genome segments with random length

Randomly select 2000 segments with windows of between 100 to 10000 bp each.

In [ ]:
rand_seg_df2 = pd.DataFrame({'Gene_Length': np.hstack([np.repeat(None, 2000)])})
rand_seg_df2['Start_pos'] = None
rand_seg_df2['End_pos'] = None
rand_seg_df2['Alignment'] = True
rand_seg_df2

In [ ]:
np.random.seed(100)

max_start_pos = bool_mat.shape[1] - 10000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    length = np.random.randint(100, 10001)  # Random length between 100 and 10000
    end_pos = start_pos + length - 1
    rand_seg_df2.loc[i, 'Gene_Length'] = length
    rand_seg_df2.loc[i, 'Start_pos'] = start_pos
    rand_seg_df2.loc[i, 'End_pos'] = end_pos

print(rand_seg_df2.head())
rand_seg_df2.to_csv("../data/klebsiella/rand_seg_info2.csv", index=False)